# Pasca-prediksi tanpa latih ulang: kalibrasi, pergeseran proporsi kelas, himpunan conformal
**Kapan dipakai:** sesudah ada OOF + prediksi test dari notebook mana pun (`baseline_oof`,
`finetune_oof`, `stacking_teks_tabular`, atau hasil `ensemble_oof`). Semua langkah hanya
memakai peluang yang sudah tersimpan, jadi cepat (detik) dan bisa diulang setelah model baru.

1. **Temperature scaling:** satu angka `T` membuat peluang jujur (syarat langkah 2 dan 3).
2. **Pergeseran prior (label shift):** proporsi kelas di test bisa beda dari train. Diperkirakan
   dengan **EM Saerens (2002)** dan **BBSE (Lipton 2018)**, lalu peluang test dikoreksi. Koreksi
   hanya dipakai kalau **simulasi di OOF** membuktikan skornya naik.
3. **Conformal prediction:** tiap baris diberi *himpunan* label yang memuat label benar ≥ 90%
   baris (dijamin secara statistik). Baris dengan himpunan > 1 label = model ragu. Bahan slide:
   "x% test ambigu antara netral dan positif".

**Data demo:** SmSA + OOF `tfidf_logreg` dari `baseline_oof` (jalankan itu dulu).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss

## Konfigurasi
`OOF_FILE` dan `TEST_FILE` = peluang (n × jumlah kelas, urutan kelas `np.unique(train[TARGET])`).

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/pasca_prediksi"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
KEY_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"                # HAPUS saat lomba (kunci demo)

ID_COL = "id"                    # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUB_COL = TARGET                 # GANTI saat lomba bila kolom prediksi di sample bernama lain
OOF_FILE = f"{OOF_FOLDER}/oof_tfidf_logreg.npy"                # GANTI: model mana pun
TEST_FILE = f"{OOF_FOLDER}/test_tfidf_logreg.npy"              # GANTI
# OOF_FILE = "../../../outputs/text/ensemble_oof/blend_oof_jujur.npy"   # alt: hasil ensemble_oof
# TEST_FILE = "../../../outputs/text/ensemble_oof/blend_test.npy"

ALPHA = 0.1          # conformal: label benar ada di himpunan untuk >= 90% baris
N_TRIALS = 30        # ulangan simulasi pergeseran prior di OOF
MIN_GAIN = 0.002     # koreksi prior dipakai hanya bila skor simulasi naik sekurangnya ini
SEED = 42


def metric(truth, labels):
    return f1_score(truth, labels, average="macro")    # GANTI: metrik lomba
    # return accuracy_score(truth, labels)


print("metrik uji:", metric(np.array([0, 1, 1]), np.array([0, 1, 0])))

## Baca data dan peluang
Kalau file berisi skor (ada nilai negatif, mis. `decision_function` SVM), diubah ke peluang
dengan softmax. Baris OOF harus sama urutannya dengan train.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
CLASSES = np.unique(train[TARGET])
y = np.searchsorted(CLASSES, train[TARGET].values)
oof = np.load(OOF_FILE).astype("float64")
test_pred = np.load(TEST_FILE).astype("float64")
if oof.min() < 0:
    oof = softmax(oof, axis=1)
    test_pred = softmax(test_pred, axis=1)
oof = oof / oof.sum(axis=1, keepdims=True)
test_pred = test_pred / test_pred.sum(axis=1, keepdims=True)
train_prior = np.bincount(y, minlength=len(CLASSES)) / len(y)
print("OOF:", oof.shape, "| test:", test_pred.shape, "| kelas:", CLASSES)
print("baris cocok:", oof.shape[0] == len(train), test_pred.shape[0] == len(test))
print("skor OOF:", round(metric(y, oof.argmax(axis=1)), 4),
      "| log loss:", round(log_loss(y, oof), 4))

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = CLASSES[0]
demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## 1. Temperature scaling
`T > 1`: model terlalu yakin (umum pada fine-tune BERT), `T < 1`: terlalu ragu (umum pada
Naive Bayes/LogReg ber-C kecil). `argmax` tidak berubah, jadi F1 tetap; yang membaik log loss.

In [ ]:
def sharpen(pred, temperature):
    return softmax(np.log(np.clip(pred, 1e-9, 1)) / temperature, axis=1)


def oof_loss(temperature):
    return log_loss(y, sharpen(oof, temperature), labels=range(len(CLASSES)))


TEMPERATURE = minimize_scalar(oof_loss, bounds=(0.2, 5), method="bounded").x
oof_cal = sharpen(oof, TEMPERATURE)
test_cal = sharpen(test_pred, TEMPERATURE)
print("T =", round(TEMPERATURE, 3), "| log loss", round(oof_loss(1.0), 4), "->",
      round(oof_loss(TEMPERATURE), 4))

## 2. Perkiraan proporsi kelas di test
**EM:** ulangi "koreksi peluang dengan prior baru → prior = rata-rata peluang" sampai stabil.
**BBSE:** pakai matriks kebingungan OOF: `P(tebak=i) di test = Σ_j P(tebak=i, benar=j) · w_j`,
selesaikan `w`, prior test = `w × prior train`. Dua cara yang setuju = perkiraan lebih bisa
dipercaya.

In [ ]:
def em_prior(proba, steps=200):
    prior = train_prior.copy()
    adjusted = proba
    for _ in range(steps):
        adjusted = proba * (prior / train_prior)
        adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
        new_prior = adjusted.mean(axis=0)
        if np.abs(new_prior - prior).max() < 1e-6:
            break
        prior = new_prior
    return prior, adjusted


def bbse_prior(proba):
    joint = confusion_matrix(y, oof_cal.argmax(axis=1), labels=range(len(CLASSES)))
    joint = joint.T / len(y)                                  # [tebakan, benar]
    guess = np.bincount(proba.argmax(axis=1), minlength=len(CLASSES)) / len(proba)
    weights = np.linalg.lstsq(joint, guess, rcond=None)[0]
    prior = np.clip(weights * train_prior, 0, None)
    return prior / prior.sum()


em_test, test_em = em_prior(test_cal)
priors = pd.DataFrame({"train": train_prior, "test_EM": em_test, "test_BBSE": bbse_prior(test_cal)},
                      index=CLASSES)
priors["tebakan_test"] = np.bincount(test_cal.argmax(axis=1), minlength=len(CLASSES)) / len(test)
priors.round(3).to_csv(f"{OUTPUT_FOLDER}/prior_kelas.csv")
print(priors.round(3))

## Simulasi jujur: apakah koreksi prior menaikkan skor?
OOF diambil ulang (bootstrap per kelas) supaya proporsinya = perkiraan EM untuk test, lalu
dibandingkan skor tanpa vs dengan koreksi EM. Dilakukan juga pada proporsi train (tanpa
pergeseran): di sana selisih harus ≈ 0. F1 makro bisa *turun* walau akurasi naik (koreksi
mendorong ke kelas mayoritas), karena itu diukur dengan `metric`.

In [ ]:
def shift_trial(prior, rng):
    rows = []
    for class_id in range(len(CLASSES)):
        pool = np.flatnonzero(y == class_id)
        rows.append(rng.choice(pool, int(round(prior[class_id] * len(y))), replace=True))
    rows = np.concatenate(rows)
    estimate, adjusted = em_prior(oof_cal[rows])
    raw = metric(y[rows], oof_cal[rows].argmax(axis=1))
    fixed = metric(y[rows], adjusted.argmax(axis=1))
    return {"galat_prior": np.abs(estimate - prior).sum() / 2, "naik_skor": fixed - raw}


rng = np.random.default_rng(SEED)
rows = []
for name, prior in {"seperti test (EM)": em_test, "seperti train": train_prior}.items():
    for _ in range(N_TRIALS):
        rows.append(shift_trial(prior, rng) | {"proporsi": name})
simulation = pd.DataFrame(rows).groupby("proporsi")
simulation = simulation.agg(["mean", "std"]).round(4)
GAIN = simulation.loc["seperti test (EM)", ("naik_skor", "mean")]
USE_PRIOR = GAIN >= MIN_GAIN
print(simulation)
print("koreksi prior dipakai:", USE_PRIOR, "| rata-rata naik skor:", GAIN)

## 3. Himpunan conformal (cakupan dijamin)
Skor ketidaksesuaian LAC = `1 − peluang label benar`. Ambang = kuantil (1 − `ALPHA`) skor di
OOF; himpunan = semua kelas dengan `1 − peluang ≤ ambang`. **Per kelas** (Mondrian): ambang
dihitung terpisah tiap kelas, sehingga kelas kecil juga tercakup ≥ 90%, dan tetap sah walau
proporsi kelas test bergeser. Cakupan diuji jujur: kalibrasi di separuh OOF, uji di separuh lain.

In [ ]:
def lac_threshold(proba, truth):
    scores = 1 - proba[np.arange(len(truth)), truth]
    level = min(np.ceil((len(scores) + 1) * (1 - ALPHA)) / len(scores), 1.0)
    return np.quantile(scores, level, method="higher")


def class_thresholds(proba, truth):
    thresholds = np.ones(len(CLASSES))
    for class_id in range(len(CLASSES)):
        rows = truth == class_id
        thresholds[class_id] = lac_threshold(proba[rows], truth[rows])
    return thresholds


def evaluate_sets(sets, truth):
    covered = sets[np.arange(len(truth)), truth]
    size = sets.sum(axis=1)
    row = {"cakupan": covered.mean(), "ukuran_rata": size.mean(), "tunggal": (size == 1).mean(),
           "kosong": (size == 0).mean()}
    for class_id, name in enumerate(CLASSES):
        row[f"cakupan_{name}"] = covered[truth == class_id].mean()
    return row


print("uji ambang:", lac_threshold(oof_cal, y).round(3), class_thresholds(oof_cal, y).round(3))

Bandingkan LAC global vs per kelas (dua kali: separuh A → B, B → A, lalu dirata-rata).

In [ ]:
half = np.random.default_rng(SEED).permutation(len(y)) % 2 == 0
rows = []
for method, fit in {"LAC global": lac_threshold, "LAC per kelas": class_thresholds}.items():
    for calib in [half, ~half]:
        thresholds = fit(oof_cal[calib], y[calib])
        sets = oof_cal[~calib] >= 1 - thresholds
        rows.append(evaluate_sets(sets, y[~calib]) | {"cara": method})
coverage = pd.DataFrame(rows).groupby("cara")
coverage = coverage.mean().round(4)
coverage.to_csv(f"{OUTPUT_FOLDER}/conformal_cakupan.csv")
print(coverage.T)

class_cols = []
for name in CLASSES:
    class_cols.append(f"cakupan_{name}")
figure, ax = plt.subplots(figsize=(10, 4))
by_class = coverage[class_cols].T
by_class.plot.bar(ax=ax, color=["#0072B2", "#E69F00"], rot=0)
ax.axhline(1 - ALPHA, color="#1a1a1a", linestyle="--", linewidth=1.5)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("cakupan (label benar ada di himpunan)")
figure.savefig(f"{OUTPUT_FOLDER}/conformal_cakupan.png", dpi=200, bbox_inches="tight")
plt.show()

## Himpunan untuk test
Ambang per kelas dari seluruh OOF, diterapkan pada peluang test terkalibrasi (tanpa koreksi
prior: ambang dihitung di skala itu). `himpunan_test.csv` = bahan analisis baris ambigu.

In [ ]:
thresholds = class_thresholds(oof_cal, y)
test_sets = test_cal >= 1 - thresholds
names = pd.Series("", index=range(len(test)))
for class_id, name in enumerate(CLASSES):
    names = names + np.where(test_sets[:, class_id], f"{name}|", "")
test_table = pd.DataFrame()
test_table[ID_COL] = test[ID_COL].values
test_table["prediksi"] = CLASSES[test_cal.argmax(axis=1)]
test_table["himpunan"] = names.str.rstrip("|").values
test_table["ukuran"] = test_sets.sum(axis=1)
test_table.to_csv(f"{OUTPUT_FOLDER}/himpunan_test.csv", index=False)
sizes = test_table["ukuran"].value_counts()
print("ukuran himpunan test:", sizes.sort_index().to_dict())
ambiguous = test_table[test_table["ukuran"] > 1]
print(f"test ambigu (> 1 label): {len(ambiguous) / len(test):.1%}")
print(ambiguous["himpunan"].value_counts().head(5))

## Simpan peluang akhir + ringkasan
`oof_pasca.npy` (terkalibrasi) dan `test_pasca.npy` (terkalibrasi, + koreksi prior kalau
lolos simulasi) di `OUTPUT_FOLDER`, tidak di `OOF_FOLDER` (supaya `ensemble_oof` tidak
membacanya sebagai model kembar). `ringkasan.csv` = angka kunci untuk slide.

In [ ]:
final_test = test_cal
if USE_PRIOR:
    final_test = test_em
np.save(f"{OUTPUT_FOLDER}/oof_pasca.npy", oof_cal.astype("float32"))
np.save(f"{OUTPUT_FOLDER}/test_pasca.npy", final_test.astype("float32"))
summary = pd.Series({"sumber": OOF_FILE, "temperature": round(TEMPERATURE, 3),
                     "skor_oof": round(metric(y, oof_cal.argmax(axis=1)), 4),
                     "naik_skor_simulasi_prior": GAIN, "koreksi_prior_dipakai": USE_PRIOR,
                     "alpha": ALPHA, "test_ambigu": round(len(ambiguous) / len(test), 4)})
summary.to_csv(f"{OUTPUT_FOLDER}/ringkasan.csv", header=["nilai"])
print(summary.to_string())

## Submission

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(CLASSES[final_test.argmax(axis=1)], index=test[ID_COL].values)

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada prediksi kosong")
foreign = set(written[SUB_COL].dropna()) - set(CLASSES.tolist())
if len(foreign) > 0:
    problems.append(f"label di luar train: {sorted(foreign)[:5]}")
print(written[SUB_COL].value_counts())
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)
Skor test tanpa vs dengan koreksi prior, dan cakupan himpunan conformal di test sungguhan.

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).values)
print("prior test sebenarnya:", (np.bincount(truth) / len(truth)).round(3))
print("skor test tanpa koreksi:", round(metric(truth, test_cal.argmax(axis=1)), 4),
      "| dengan koreksi EM:", round(metric(truth, test_em.argmax(axis=1)), 4))
print("akurasi tanpa/dengan:", round(accuracy_score(truth, test_cal.argmax(axis=1)), 4),
      round(accuracy_score(truth, test_em.argmax(axis=1)), 4))
test_coverage = pd.Series(evaluate_sets(test_sets, truth))
print(test_coverage.round(4).to_string())